# Preamble

In [ ]:
import matplotlib.pylab as plt
import matplotlib.gridspec as gridspec
import xarray as xr
import pandas as pd
from ocean_coordinates import choose_region

# Data

In [ ]:
seasons_df = pd.read_csv('top_of_ABL_values.csv', index_col=0)
seasons_df

In [ ]:
height_in_meters = xr.open_dataset("00_data/00_grid_info/vertical_info/mean_cell_altitude.nc")
height_in_meters;

# Statistics

In [ ]:
def map_altitude_to_meters(seasons_df, height_ds):
    """
    Replace altitude indices in seasons_df with heights in meters
    using the basin-specific mappings from an xarray Dataset.
    """

    seasons_df_in_meter = seasons_df.copy().astype(float)

    for basin in seasons_df.index:
        if basin not in height_ds:
            raise ValueError(f"{basin} not found in height dataset")

        heights = height_ds[basin].values
        
        seasons_df_in_meter.loc[basin] = seasons_df.loc[basin].apply(
            lambda alt_idx: heights[int(alt_idx)]
        )

    return seasons_df_in_meter

In [ ]:
seasons_df_in_meters = map_altitude_to_meters(seasons_df, height_in_meters)
seasons_df_in_meters

In [ ]:
overall_mean = seasons_df_in_meters.mean().mean()
overall_std = seasons_df_in_meters.stack().std()

print(f"Overall Mean: {overall_mean:.2f} meters")
print(f"Overall Standard Deviation: {overall_std:.2f} meters")

In [ ]:
total_values = seasons_df_in_meters.size
below_1000m_count = (seasons_df_in_meters < overall_mean).sum().sum()
below_1000m_percentage = (below_1000m_count / total_values) * 100

print(f"Percentage of values below {overall_mean} meters: {below_1000m_percentage:.2f}%")

# Plotting

In [ ]:
SIZE = 25
plt.rcParams['axes.labelsize']  = SIZE
plt.rcParams['legend.fontsize'] = SIZE
plt.rcParams['xtick.labelsize'] = SIZE
plt.rcParams['ytick.labelsize'] = SIZE
plt.rcParams['font.size']       = SIZE

In [ ]:
fig = plt.figure(figsize=(18,8), facecolor='w', edgecolor='k')
G = gridspec.GridSpec(1,1)

ax1 = plt.subplot(G[0,0])
ax22 = ax1.twinx()

YLIM_TOP = 2500

ax1.axhline(1000, color='black', lw=1, ls='dashed')
ax1.axhline(2000, color='black', lw=1, ls='dashed')
ax1.axhline(3000, color='black', lw=1, ls='dashed')

ax1.plot(height_in_meters.atlantic[seasons_df.loc['atlantic'].values], marker='o', ms=15, lw=3, color='#E4572E', label='Atlantic', clip_on=False)
ax1.plot(height_in_meters.indian_ocean[seasons_df.loc['indian_ocean'].values], marker='o', ms=15, lw=3, color='#17BEBB', label='Indian Ocean', zorder=10, clip_on=False)
ax1.plot(height_in_meters.western_pacific[seasons_df.loc['western_pacific'].values], marker='o', ms=15, lw=3, color='#FFC914', label='Western Pacific', clip_on=False, zorder=12)
ax1.plot(height_in_meters.central_pacific[seasons_df.loc['central_pacific'].values], marker='o', ms=15, lw=3, color='#2E282A', label='Central Pacific', clip_on=False, zorder=14)
ax1.plot(height_in_meters.eastern_pacific[seasons_df.loc['eastern_pacific'].values], marker='o', ms=15, lw=3, color='#76B041', label='Eastern Pacific', clip_on=False)

ax1.spines[['top','right']].set_visible(False)
ax1.spines[['bottom', 'left']].set_position(('outward',20))
ax1.set_xlim(0,7)
ax1.set_xticklabels(["MAM-'20", "JJA-'20", "SON-'20", "DJF-'20", "MAM-'21", "JJA-'21", "SON-'21", "DJF-'21",''])
ax1.set_ylim(0,YLIM_TOP)
ax1.set_yticks([0,500,1000,1500,2000,2500])

ax1.set_xlabel('Season')
ax1.set_ylabel('Altitude / m')
    
ax22.set_ylim(0,YLIM_TOP)
ax22.spines[['left', 'bottom','top','right']].set_visible(False)
ax22.spines[['right']].set_position(('outward',20))
ax22.tick_params(axis='y', colors='black')

ax22.axhline(overall_mean, color='black', lw=3, ls='dashdot')
ax22.set_yticks([overall_mean])
ax22.set_yticklabels([f"{overall_mean:.0f}m"+r" ($\overline{H_0}$)"])

fig.legend(loc='upper center', bbox_to_anchor=(0.5, 0), fancybox=True, shadow=False, ncol=8)

plt.tight_layout()

filename = f'fig_03.png'
filepath = '01_figs/'
plt.savefig(filepath + filename, facecolor='white', bbox_inches='tight', dpi=400)

plt.show()